<br>

<img src="https://lindas.admin.ch/static-assets/img/lindaslogo.png" style="width:15%; float:right">

# Historisiertes Gemeindeverzeichnis - Linked Data Tutorial

## Einführung

Dieses Tutorial ist dazu gedacht, eine Einführung in das historisierte Gemeindeverzeichnis in Linked Data zu geben. Dabei soll einerseits das verwendete Datenmodell vorgestellt werden und anderseits aufgezeigt werden, wie mit Hilfe von SPARQL Abfragen die Daten genutzt werden können.

## Das historisierte Gemeindeverzeichnis

Gemeinden in der Schweiz unterliegen Veränderungen. Aktuell sind das primär Fusionen zwischen einzelnen Gemeinden. Das historisierte Gemeindeverzeichnis gibt sowohl einen Überblick über alle aktuell existierenden Gemeinden als auch über Gemeinden im Verlaufe der Zeit. Im historischen Gemeindeverzeichnis sind also auch solche Gemeinden zu finden, die heute nicht mehr existieren, weil sie fusioniert haben.

## Zielpublikum

Das Zielpublikum dieses Tutorials sind Personen, die beruflich mit Gemeinde Daten zu tun haben und über gute Grundkenntnisse in der Informatik verfügen, die aber noch wenig Knowhow zum Thema Linked Data und SPARQL besitzen.

## Interaktives Notebook

Dieses Tutorial ist ein sogenanntes **interaktives JupyterLite Notebook**. In diesem Notebook können Sie den Inhalt der einzelnen Zellen interaktiv ändern und diese Zellen direkt ausführen, um das Ergebnis Ihrer Änderungen sofort zu sehen. Die Zellen enthalten entweder [Markdown](https://en.wikipedia.org/wiki/Markdown)-Inhalt (wie diese Zelle) oder ausführbaren Python-Quellcode. Dies ist für ein Tutorial sehr hilfreich, weil Inhalte beliebig mit ausführbarem Quellcode kombiniert werden können. Es können also Abfragen gezeigt werden, diese erklärt werden und darauf weiter aufgebaut werden.

**Um direkt loslegen zu können klicken Sie oben im Menu auf Run -> Run All Cells.**  
**Einzelne ausgewählte Zellen können sie danach abändern und mit dem "Play-Button" erneut ausführen und so Abfragen individuell anpassen.**

Das Notebook startet mit einem [Setup](#Setup) der Programierumgebung. Das eigentliche Tutorial startet [hier](#Tutorial).

*Zusätzliche Informationen zu JupyterLite:*  
JupyterLite is ein spezielles Jupyter Notebook mit dem Vorteil, vollständig browserbasiert zu sein, ohne eine aufwändige Backend-Infrastruktur zu benötigen. Der Nachteil ist, dass die erstmalige Ausführung der Zellen einige Zeit in Anspruch nehmen kann, weil eine erhebliche Menge von Daten geladen werden muss. Dass eine Zelle noch in Ausführung ist, ist am `[*]` links neben der Zelle erkennbar. Nach Abschluss der Ausführung erscheint statt eines `*` eine Zahl. Vor der ersten Ausführung ist eine leere Klammer `[ ]` zu sehen. Nachfolgende wiederholte Ausführungen von Zellen werden aufgrund der gespeicherten Daten in Ihrem Browser-Cache viel schneller sein. Wenn Sie Änderungen am Tutorial vornehmen, werden diese Änderungen automatisch lokal im Browser-Cache gespeichert. Bei einem darauffolgenden Öffnen werden wiederum die Änderungen aus dem Browser-Cache geladen. Um wieder zur Ursprüngsversion des Tutorials zurückzukehren, müssen entweder die Browser-Daten gelöscht werden, oder Sie öffnen das Tutorial in einem Chrome-Inkognito Fenster, wo die Änderungen nach Schliessen des Fensters aus dem Cache gelöscht werden.

Wenn Sie mehr über die Handhabung von Jupyter Notebooks wissen wollen, finden Sie hier zwei nützliche Ressourcen:

- [Die JupyterLab Interface](https://jupyterlab.readthedocs.io/en/stable/user/interface.html)
- [Das Jupyter Notebook](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html)

## Setup

Eine SPARQL Abfrage ist nichts anderes als ein POST-Request an den entsprechenden Triple Store Datenbank Server. Um diese Requests und die erhaltenen Antworten einfacher handhaben zu können, enthält dieses Notebook vorbereiteten Python Code, der nachfolgend importiert wird. Zusätzlich wird das `pandas` Modul importiert, welches die Möglichkeit bietet, die tabellarischen Daten der SPARQL Abfragen als [Pandas Dataframes](https://pandas.pydata.org/docs/index.html) weiter zu verarbeiten. 

In [1]:
import pandas as pd
from ext.sparql import query, display_result

# Tutorial

## Linked Data Einführung

Linked Data beschreibt ein **Framework für den Umgang mit Daten**, die sowohl für Menschen nützlich sein sollen, als auch maschinenlesbar sind inkl. einer von Computern verarbeitbaren Semantik. Also sowohl Menschen als auch Computer sollen die Daten "verstehen" und interpretieren können. 

### RDF

Das für Linked Data verwendete Datenformat ist RDF (Resource Description Framework). Das bedeutet, dass die Daten nicht als Tabellen (wie beispielsweise in relationalen Datenbanken) sondern als **Triples** abgespeichert werden. Triples folgen der grammatikalischen Struktur **Subjekt -> Prädikat -> Objekt** und können auch als grammatikalischer Satz verstanden werden. 

Die Information "**Der Apfel ist grün**" wird also mit dem Tripel **Apfel -> ist -> grün** ausgedrückt. Alle Teile eines Triples sind dabei durch weitere Eigenschaften definiert und beschrieben die wiederum in Form von Triples beschrieben sind. Diese vielseitigen Verknüpfungen führen zu einer Netzwerkstruktur, zu einem sogenannten Graphen.

Nachfolgendes Bild aus dem [W3C Primer für RDF](https://www.w3.org/TR/rdf11-primer/) veranschaulicht diese Zusammenhänge:

![](https://www.w3.org/TR/rdf11-primer/example-graph.jpg)

### URI

Eine weitere wichtige Eigenschaft von Linked Data ist, dass alle Teile eines Triples, also Subjekt, Prädikat und Objekt weltweit eindeutig identifizierbar sind. Dafür werden URI (Universal Resource Identifier) eingesetzt. Die URI *xy* beispielsweise ist der weltweit eindeutige Identifier für die Stadt Bern. Typischerweise lassen sich URI **dereferenzieren**, das heisst, ein Request auf die entsprechende URI (bspw. in dem man sie in die Adresszeile eines Browsers eintippt) führt zu einer Website, die Infos zur entsprechenden URI enthält. Im Falle der URI der Stadt Bern wird man auf eine Webpage weitergeleitet, die diverse Informationen zur Stadt Bern enthält.

### Weitere Informationen zu Linked Data

Wer vertieft in das Thema Linked Data einsteigen möchte, dem sei beispielsweise [diese Youtube Playlist](https://www.youtube.com/watch?v=ON0wf0SEPx8&list=PLoOmvuyo5UAfY6jb46jCpMoqb-dbVewxg) empfohlen.

URI können grundsätzlich sprechend oder nicht sprechend aufgebaut sein. Sprechend heisst in diesem Zusammenhang, dass die URI schon etwas über das entsprechende Objekt kodieren. Nicht sprechende URI sind typischerweise als beliebige Folgend von Zahlen und Buchstaben aufgebaut. Die URI von Fedlex sind sprechend, sie beginnen alle mit `https://fedlex.data.admin.ch/eli/`. Die Einträge für die AS beginnen mit `https://fedlex.data.admin.ch/eli/oc/` und diejenigen für die SR mit `https://fedlex.data.admin.ch/eli/cc/`. Der weitere Aufbau der URI ist [hier](https://fedlex.data.admin.ch/de-CH/home/convention) erklärt.

### Prefixes

URIs sind analog zu Webadressen aufgebaut. Um die Lesbarkeit zu verbessern, können Abkürzungen, sogenannte **Prefixes** definiert werden, die häufig genutzte Teile einer URI zusammenfassen, bspw. `fedlex` als Prefix für `https://fedlex.data.admin.ch/eli/`. Aus `https://fedlex.data.admin.ch/eli/cc/1999/404` wird dann `fedlex:cc/1999/404`.

Für die weiteren Teile des Tutorials definieren wir folgenden Prefixes:
* `fedlex` für `https://fedlex.data.admin.ch/eli/`; Namensraum für AS und SR Einträge
* `jolux` für `http://data.legilux.public.lu/resource/ontology/jolux#`; Vokabular für das JOLux Datenmodell
* `skos` für `http://www.w3.org/2004/02/skos/core#`; Externes Vokabular
* `rdf` für `http://www.w3.org/1999/02/22-rdf-syntax-ns#`; Externes Vokabular

Mehr zum Aufbau der URIs auf Fedlex finden Sie [hier](https://fedlex.data.admin.ch/de-CH/home/convention).

## Das Datenmodell erforschen und Daten abfragen

Um die gespeicherten Daten bspw. zur Bundesverfassung zu erhalten, gibt es drei Möglichkeiten:

- Die HTML Version (dereferenzierte URI) unter https://www.fedlex.admin.ch/eli/cc/1999/404/de
- Den Metadaten-Explorer unter https://fedlex.data.admin.ch/de-CH/metadata?value=https://fedlex.data.admin.ch/eli/cc/1999/404
- Die Abfrage der weiteren Daten mit SPARQL, einer Sprache zum Abfragen von Linked Data Datenbanken (so genannten Triple Stores), ähnlich zu SQL.

Die Website https://fedlex.admin.ch und alle dort angzeigten Daten basieren auf den Linked Data aus dem Triple Store. Informationen, die auf den HTML-Seiten von Fedlex zu sehen sind, sind also auch maschinenlesbar via SPARQL Abfrage verfügbar. Nachfolgend soll Schritt für Schritt das Datenmodell von Fedlex eingeführt werden und die entsprechenden SPARQL Queries aufgezeigt werden, um auf diese Daten maschinenlesbar zugreifen zu können.

### SPARQL <a class="anchor" id="SPARQL"></a>

SPARQL ist eine Query-Sprache für Linked Data Triple Stores. Für eine allgemeine Einführung in SPARQL siehe z.B.: https://jena.apache.org/tutorials/sparql.html

Abfragen (engl. Queries) können entweder direkt über das [SPARQL Web-Interface von Fedlex](https://fedlex.data.admin.ch/de-CH/sparql) eingegeben und ausgeführt werden, oder als HTTP-POST Request an den SPARQL-Endpoint (https://fedlex.data.admin.ch/sparqlendpoint) gesendet werden.

Die letztere Methode erlaubt es, eigene Anwendungen zu bauen, die automatisch aktuelle Daten von Fedlex abfragen können. Für dieses Tutorial verwenden wir diese Methode. Die Abfragen sind jedoch in beiden Fällen identisch.

Um im Tutorial eine neue SPARQL Abfrage zu erstellen, erzeugen Sie eine neue Zelle für Code ("Plus-Zeichen" in der Titelzeile des aktuellen Tabs drücken und im Dropdown "Code" auswählen. Danach können sie mit dem Python Befehl `await query(query_string, triple_store)` die Abfrage ausführen, welche als Resultat ein Pandas Dataframe zurückgibt, welches sinnvollerweise einer Variable zugewiesen wird. Das Schlüsselwort `await` ist notwendig, weil die Abfrage asynchronen Code enthält. Die Anzeige des Dataframes erfolgt sinnvollerweise mit dem Befehl `display_result(df)`, welcher daführ sorgt, dass die URI im Dataframe als klickbare Links dargestellt werden. Die dreifachen Anführungszeichen des `query_string` ermöglichen, den String über mehrere Zeilen umzubrechen und damit eine übersichtliche Darstellung der Query zu erreichen. Um die Daten aus dem Fedlex Triple Store zu beziehen, muss für `triple_store` ein "F" Charakter übergeben werden.

Soll eine Query aus dem Tutorial über das SPARQL Web-Interface ausgeführt werden, kopieren Sie einfach den Teil zwischen den `"""` und fügen sie im [SPARQL Web-Interface](https://fedlex.data.admin.ch/de-CH/sparql) ein.

### Pattern Matching

SPARQL Queries sind Aufträge an den Computer, bestimme Muster (Pattern) in den Daten zu finden (matching). Es können also mit Hilfe von SPARQL Muster vorgegeben werden, und die Datenbank gibt alle Triples zurück, die dieses Muster erfüllen. Einzelne Positionen der Triples können dabei bei einer Abfrage bewusst undefiniert gelassen und mit einer Variable bezeichnet werden. Variablen beginnen mit `?` und werden bei der Abfrage durch alle möglichen Elemente gefüllt, die dieses Pattern erfüllen. Eine ausführlicher Anleitung zum Pattern Matching ist [hier](https://programminghistorian.org/en/lessons/retired/graph-databases-and-SPARQL#rdf-in-brief) zu finden. 

### Die erste SPARQL Query

In [2]:
df = await query("""


""", "F")

display_result(df)

<class 'TypeError'>: can only concatenate str (not "int") to str